# Train and inspect a generated-C model with `cke.v8`

This notebook is the complete bounded v8 training starter. It authors a four-layer Qwen3-style dense/GQA model with `cke.nn`, trains a BPE tokenizer from the training split, executes generated-C forward/backward and native AdamW, compares the same serialized batches with an independent PyTorch oracle, resumes a complete checkpoint in a fresh process, and exports through independently generated v8 inference.

Python controls the experiment and renders evidence. It does **not** replace missing model arithmetic or silently fall back to PyTorch. The default notebook run stops after capability preflight; enable training explicitly in the configuration cell.


## 1. Import CKE and choose the run mode

Launch Jupyter from the repository root. `RUN_DIR` can point at another local artifact directory. A report loaded with `LOAD_EXISTING_REPORT=True` is clearly labeled historical; only `experiment.run()` creates fresh identity-bound certification evidence.


In [ ]:
from pathlib import Path
import html
import json
import math
import os
import sys
from urllib.parse import quote

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "version" / "v8").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "version" / "v8").is_dir():
    raise RuntimeError("Launch this notebook from inside a C-Kernel-Engine checkout")

sys.path.insert(0, str(ROOT / "version" / "v7"))
import ckernel_engine as cke

try:
    from IPython.display import HTML, IFrame, display
except ImportError:
    class HTML(str):
        pass
    class IFrame(str):
        def __new__(cls, src, width=None, height=None):
            return super().__new__(cls, src)
    def display(value):
        print(value)

RUN_DIR = Path(os.environ.get(
    "CKE_NOTEBOOK_RUN_DIR",
    ROOT / "version" / "v8" / ".cache" / "python_authoring" / "english_fixture",
)).expanduser().resolve()
EXECUTE_TRAINING = os.environ.get("CKE_NOTEBOOK_EXECUTE", "0") == "1"
LOAD_EXISTING_REPORT = os.environ.get("CKE_NOTEBOOK_LOAD_EXISTING", "0") == "1"
EMBED_IR_VISUALIZER = os.environ.get("CKE_NOTEBOOK_EMBED_IR", "0") == "1"

print("CKE checkout:", ROOT)
print("run directory:", RUN_DIR)
print("execute generated training:", EXECUTE_TRAINING)


## 2. Declare the dataset, tokenizer, optimizer, and model

The BPE tokenizer is fitted only on the pinned training split. CKE and PyTorch then consume the same serialized token IDs, labels, masks, and batch order. The short token limits and two epochs keep this notebook practical; raise them only after the starter passes.


In [ ]:
dataset = cke.v8.DatasetConfig(
    corpus=ROOT / "version" / "v8" / "training" / "english_byte_v1.json",
    max_train_tokens=320,
    max_validation_tokens=64,
)
tokenizer = cke.v8.TokenizerConfig(
    kind="bpe",
    vocab_size=384,
    min_frequency=2,
    max_piece_bytes=24,
)
training = cke.v8.TrainingConfig(
    epochs=2,
    grad_accum=2,
    learning_rate=3e-4,
)
model = cke.models.qwen3_tiny(
    vocab=tokenizer.vocab_size,
    dim=32,
    layers=4,
    hidden=64,
    heads=4,
    kv_heads=2,
    context_len=32,
    rope_theta=1_000_000.0,
    init="normal_0p02",
    dtype="float32",
    name="v8_notebook_english_fixture",
)
experiment = cke.v8.compile(
    model,
    run_name="v8-python-english-fixture",
    run_dir=RUN_DIR,
    dataset=dataset,
    tokenizer=tokenizer,
    training=training,
)

print(model)
print(f"trainable parameters: {model.parameter_count():,}")
print("generated workflow command:")
print(" ".join(experiment.command()))


## 3. Inspect the authored semantic graph

This is the `cke.nn` graph the adapter accepts. Unsupported semantics, child-module edits, dtypes, or depths fail closed during `cke.v8.compile`; the adapter does not approximate them with a nearby preset.


In [ ]:
graph_document = experiment.graph.to_dict()
print(experiment.graph.to_markdown())
print("graph schema:", graph_document.get("schema"))
print("experiment definition:", experiment.experiment_path)


## 4. Preflight forward and backward capabilities

Preflight lists candidate providers from the canonical v8 kernel maps. It is an inventory, not a numerical PASS. Provider resolution and executed PyTorch evidence become PASS only after the generated workflow publishes a fresh, matching report.


In [ ]:
preflight = experiment.preflight()
capability_rows = []
for requirement in preflight["capabilities"]:
    candidates = requirement.get("candidates", [])
    capability_rows.append({
        "requirement": requirement["label"],
        "direction": requirement["direction"],
        "providers": ", ".join(row["provider"] for row in candidates),
        "state": "candidate" if candidates else "missing",
    })

def show_table(rows, columns=None, limit=None):
    rows = list(rows)
    if limit is not None:
        rows = rows[:limit]
    if not rows:
        display(HTML("<p><em>No rows available.</em></p>"))
        return
    columns = list(columns or rows[0].keys())
    head = "".join(f"<th>{html.escape(str(column))}</th>" for column in columns)
    body = "".join(
        "<tr>" + "".join(f"<td><code>{html.escape(str(row.get(column, '')))}</code></td>" for column in columns) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>"))

print(preflight["status"], "can_launch_generated_workflow=", preflight["can_launch_generated_workflow"])
show_table(capability_rows, ["requirement", "direction", "providers", "state"])


## 5. Run generated-C training or load an earlier report

Set `EXECUTE_TRAINING=True` in the first cell, or launch with `CKE_NOTEBOOK_EXECUTE=1`, to run the full workflow. Compilation and the independent oracle make this much slower than preflight. `report_source` distinguishes fresh certification from an older report opened for exploration.


In [ ]:
report = None
report_source = "none"
if EXECUTE_TRAINING:
    report = experiment.run()
    report_source = "fresh identity-bound execution"
elif LOAD_EXISTING_REPORT and experiment.report_path.is_file():
    report = json.loads(experiment.report_path.read_text(encoding="utf-8"))
    report_source = "historical artifact (not certified by this notebook invocation)"

print("report source:", report_source)
print("report path:", experiment.report_path)
if report is None:
    print("No training report loaded. Preflight is complete; enable execution or historical loading to inspect results.")
else:
    print("workflow status:", report.get("status"), "passed=", report.get("passed"))


## 6. Inspect the tokenizer and exact token stream

After execution, these artifacts show the fitted tokenizer contract, sample token IDs, exact decode checks, split lineage, and dataset quality gates. They are the inputs shared by CKE and PyTorch.


In [ ]:
def read_json(path):
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8")) if path.is_file() else None

roundtrip = read_json(RUN_DIR / "tokenizer_roundtrip.json")
profile = read_json(RUN_DIR / "dataset_profile.json")
quality = read_json(RUN_DIR / "tokenizer_quality_gate.json")
if roundtrip:
    print("tokenizer exact round trip:", roundtrip.get("exact_match"))
    show_table(roundtrip.get("sample_rows", []), ["line_no", "token_count", "token_ids", "decoded", "exact_match"])
else:
    print("Tokenizer artifacts appear after generated training runs.")
if profile:
    print("split profile:")
    show_table([{"split": name, **row} for name, row in profile.get("splits", {}).items()],
               ["split", "tokens", "source_revision", "token_ids_sha256"])
if quality:
    print("tokenizer quality verdict:", quality.get("verdict", quality.get("status")))


## 7. View training and held-out loss curves

The chart is rendered as inline SVG and needs no plotting package. CKE and PyTorch losses come from the same batches; held-out loss is evaluated before and after training.


In [ ]:
def line_chart(series, *, title, x_label, y_label, log_y=False, width=760, height=330):
    points = [(name, [(float(x), float(y)) for x, y in values if y is not None and math.isfinite(float(y))])
              for name, values in series.items()]
    points = [(name, values) for name, values in points if values]
    if not points:
        display(HTML("<p><em>No chart data available.</em></p>"))
        return
    transformed = [(name, [(x, math.log10(max(y, 1e-30)) if log_y else y) for x, y in values])
                   for name, values in points]
    xs = [x for _name, values in transformed for x, _y in values]
    ys = [y for _name, values in transformed for _x, y in values]
    xmin, xmax = min(xs), max(xs); ymin, ymax = min(ys), max(ys)
    if xmax == xmin: xmax = xmin + 1.0
    if ymax == ymin: ymax = ymin + 1.0
    left, right, top, bottom = 68, 20, 38, 52
    plot_w, plot_h = width - left - right, height - top - bottom
    sx = lambda x: left + (x - xmin) / (xmax - xmin) * plot_w
    sy = lambda y: top + plot_h - (y - ymin) / (ymax - ymin) * plot_h
    colors = ["#2563eb", "#dc2626", "#059669", "#7c3aed", "#d97706"]
    svg = [f'<svg viewBox="0 0 {width} {height}" width="100%" role="img" aria-label="{html.escape(title)}">',
           '<rect width="100%" height="100%" fill="#fff"/>',
           f'<text x="{left}" y="22" font-size="16" font-weight="600">{html.escape(title)}</text>',
           f'<line x1="{left}" y1="{top}" x2="{left}" y2="{top+plot_h}" stroke="#64748b"/>',
           f'<line x1="{left}" y1="{top+plot_h}" x2="{left+plot_w}" y2="{top+plot_h}" stroke="#64748b"/>']
    for tick in range(5):
        value = ymin + (ymax-ymin)*tick/4
        y = sy(value)
        label = f"10^{value:.1f}" if log_y else f"{value:.4g}"
        svg += [f'<line x1="{left}" y1="{y:.1f}" x2="{left+plot_w}" y2="{y:.1f}" stroke="#e2e8f0"/>',
                f'<text x="{left-8}" y="{y+4:.1f}" text-anchor="end" font-size="11">{label}</text>']
    for index, (name, values) in enumerate(transformed):
        color = colors[index % len(colors)]
        path = " ".join(("M" if i == 0 else "L") + f" {sx(x):.1f} {sy(y):.1f}" for i, (x, y) in enumerate(values))
        svg.append(f'<path d="{path}" fill="none" stroke="{color}" stroke-width="2"/>')
        lx = left + index * 150
        svg += [f'<line x1="{lx}" y1="{height-15}" x2="{lx+22}" y2="{height-15}" stroke="{color}" stroke-width="3"/>',
                f'<text x="{lx+28}" y="{height-11}" font-size="11">{html.escape(name)}</text>']
    svg += [f'<text x="{left+plot_w/2}" y="{height-30}" text-anchor="middle" font-size="12">{html.escape(x_label)}</text>',
            f'<text x="15" y="{top+plot_h/2}" transform="rotate(-90 15 {top+plot_h/2})" text-anchor="middle" font-size="12">{html.escape(y_label)}</text>',
            '</svg>']
    display(HTML("".join(svg)))

learning = report.get("checks", {}).get("learning", {}) if report else {}
epochs = learning.get("epochs", [])
line_chart({
    "CKE generated C": [(row["epoch"], row.get("cke_mean_loss")) for row in epochs],
    "PyTorch oracle": [(row["epoch"], row.get("pytorch_mean_loss")) for row in epochs],
}, title="Training loss by epoch", x_label="epoch", y_label="mean token loss")
if learning:
    show_table([{
        "training_first": learning.get("first_epoch_loss"),
        "training_last": learning.get("last_epoch_loss"),
        "heldout_before": learning.get("heldout_loss_before"),
        "heldout_after": learning.get("heldout_loss_after"),
        "passed": learning.get("passed"),
    }])


## 8. Review PyTorch parity through the optimizer trajectory

Loss alone is not the oracle. This view checks gradients, AdamW moments, updated weights, selected logits, and the final partial accumulation window against declared elementwise tolerances.


In [ ]:
parity = report.get("checks", {}).get("pytorch_trajectory", {}) if report else {}
tolerances = parity.get("tolerances", {})
parity_metrics = [
    {"quantity": "weights", "max_abs_diff": parity.get("max_weight_abs_diff"), "tolerance": tolerances.get("parameter")},
    {"quantity": "AdamW moments", "max_abs_diff": parity.get("max_moment_abs_diff"), "tolerance": tolerances.get("moment")},
    {"quantity": "gradients", "max_abs_diff": parity.get("max_gradient_abs_diff"), "tolerance": tolerances.get("gradient")},
    {"quantity": "loss", "max_abs_diff": parity.get("max_loss_abs_diff"), "tolerance": tolerances.get("loss")},
    {"quantity": "selected logits", "max_abs_diff": parity.get("max_selected_logits_abs_diff"), "tolerance": tolerances.get("logits")},
]
for row in parity_metrics:
    value, tolerance = row["max_abs_diff"], row["tolerance"]
    row["within_contract"] = value is not None and tolerance is not None and value <= tolerance
show_table(parity_metrics, ["quantity", "max_abs_diff", "tolerance", "within_contract"])

trajectory = parity.get("trajectory", [])
line_chart({
    "parameter diff": [(row["step"], row.get("max_param_diff")) for row in trajectory],
    "gradient diff": [(row["step"], row.get("gradient_max_abs_diff")) for row in trajectory],
    "moment diff": [(row["step"], row.get("moment_max_abs_diff")) for row in trajectory],
    "loss diff": [(row["step"], row.get("loss_diff")) for row in trajectory],
}, title="CKE versus PyTorch discrepancy", x_label="optimizer step", y_label="max absolute difference (log scale)", log_y=True)

tensor_diffs = parity.get("final_tensor_discrepancies", {})
for group in ("weights", "optimizer", "partial_window_gradients"):
    rows = sorted(tensor_diffs.get(group, []), key=lambda row: row.get("max_abs_diff", 0.0), reverse=True)
    if rows:
        print(group, "— largest final discrepancies")
        show_table(rows, limit=8)


## 9. Verify checkpoint/resume and inference export

Same-runtime fresh-process resume is held to exact state equality. Deployment evidence rebuilds v8 inference independently, compares logits, and checks a multi-token generated sequence. A standalone native executable remains explicitly uncertified in this starter.


In [ ]:
checks = report.get("checks", {}) if report else {}
resume = checks.get("fresh_process_resume", {})
checkpoint = checks.get("checkpoint_compatibility", {})
export = checks.get("inference_export", {})
summary_rows = [
    {"gate": "fresh-process resume", "passed": resume.get("passed"),
     "evidence": f"weight diff={resume.get('weight_max_abs_diff')}, optimizer diff={resume.get('optimizer_max_abs_diff')}"},
    {"gate": "checkpoint identity controls", "passed": checkpoint.get("passed"),
     "evidence": checkpoint.get("authoritative_identity")},
    {"gate": "independent v8 inference export", "passed": export.get("passed"),
     "evidence": f"last-token diff={export.get('v8_inference_last_token_logits_max_abs_diff')}"},
    {"gate": "generated token trajectory", "passed": export.get("generation_tokens_match"),
     "evidence": f"steps={export.get('generation_steps')}, logits diff={export.get('generation_logits_max_abs_diff')}"},
    {"gate": "standalone native executable", "passed": False,
     "evidence": export.get("standalone_native_executable", "NOT_CERTIFIED")},
]
show_table(summary_rows, ["gate", "passed", "evidence"])
if export.get("generation_sequence"):
    print("generated token IDs:", export["generation_sequence"])


## 10. Open the training IR visualizer

`ir_report.html` contains the dataset and tokenizer views, forward/backward operation graph, saved tensors, gradient stitching, memory plan, checkpoint evidence, parity, learning, and export panels. The JSON report remains the certification verdict; the presence of HTML alone is not a PASS.


In [ ]:
visualizer_path = Path(
    (report or {}).get("artifacts", {}).get("ir_visualizer", RUN_DIR / "ir_report.html")
).resolve()
if visualizer_path.is_file():
    try:
        relative_visualizer = visualizer_path.relative_to(ROOT).as_posix()
        visualizer_url = "/files/" + quote(relative_visualizer)
    except ValueError:
        visualizer_url = visualizer_path.as_uri()
    display(HTML(
        f'<p><a href="{html.escape(visualizer_url)}" target="_blank" rel="noopener">'
        f'Open the interactive training IR visualizer</a><br><code>{html.escape(str(visualizer_path))}</code></p>'
    ))
    if EMBED_IR_VISUALIZER:
        display(IFrame(src=visualizer_url, width="100%", height=760))
else:
    print("IR visualizer is not available yet:", visualizer_path)


## 11. Locate the complete evidence bundle

Keep the run directory with the experiment definition when sharing a result. The invocation ID, configuration hash, corpus/token-stream hashes, loaded-library provenance, checkpoint, generated sources, and inference probe together establish what actually ran.


In [ ]:
artifact_candidates = {
    "experiment definition": experiment.experiment_path,
    "capability preflight": experiment.preflight_path,
    "workflow verdict": experiment.report_path,
    "tokenizer": RUN_DIR / "tokenizer.json",
    "serialized training tokens": Path(
        (report or {}).get("corpus", {}).get("splits", {}).get("train", {}).get(
            "token_ids", RUN_DIR / "dataset" / "train_token_ids.i32"
        )
    ),
    "loss curve": RUN_DIR / "training_loss_curve_latest.json",
    "parity trajectory": RUN_DIR / "training_parity_latest.json",
    "checkpoint policy": RUN_DIR / "training_checkpoint_policy_latest.json",
    "training IR visualizer": RUN_DIR / "ir_report.html",
}
if report:
    for name, path in report.get("artifacts", {}).items():
        if isinstance(path, str):
            artifact_candidates[name.replace("_", " ")] = Path(path)
artifact_rows = []
for name, path in artifact_candidates.items():
    path = Path(path)
    artifact_rows.append({"artifact": name, "present": path.exists(), "path": str(path)})
show_table(artifact_rows, ["artifact", "present", "path"])


### Reading the result

- `RESOLVED_EXECUTION_PASS` means a fresh generated workflow matched the authored experiment and completed its numerical, restart, and export gates.
- Similar loss curves support learning behavior, while gradient, moment, weight, and logit comparisons establish the numerical contract.
- Historical reports are useful for inspection but do not certify the current notebook invocation.
- This starter certifies the declared reduced FP32 dense/GQA circuit. It does not claim arbitrary `torch.nn`, BF16, LoRA/QLoRA, Qwen3.5 DeltaNet, or RWKV support.
